# MenoPhenotype — Notebook 4: Train-Test Split

**Course stage covered:** Stage 9 (Train-Test Split).
**Input:** `artifacts/baseline_engineered.csv`, `artifacts/baseline_clean.csv` (for `STATUS0`),
`artifacts/screener_engineered.csv`, `artifacts/screener_clean.csv`.
**Output:** `artifacts/split_ids.json` (train/test SWANIDs), `artifacts/baseline_posthoc_labels.csv`,
`artifacts/screener_validation_ids.csv`, `artifacts/screener_validation_posthoc.csv`.

This runs **before** any data-dependent preprocessing (Notebook 5), per the brief's Section 15: no scaler,
imputer, or feature-selection statistic may be fit before this split exists.


In [1]:
import json
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

RNG = 42
ART = Path('artifacts')
feat = pd.read_csv(ART / 'baseline_engineered.csv').set_index('SWANID')
df = pd.read_csv(ART / 'baseline_clean.csv').set_index('SWANID')
scr = pd.read_csv(ART / 'screener_clean.csv').set_index('SWANID')
scr_feat = pd.read_csv(ART / 'screener_engineered.csv').set_index('SWANID')
print(feat.shape, scr_feat.shape)


/tmp/ipykernel_215/3609467127.py:9: DtypeWarning: Columns (0: AIDSMED0, 1: DELVR100, 2: READSPE0, 3: THNKSPE0, 4: BADLUCK0, 5: EXPERTS0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ART / 'baseline_clean.csv').set_index('SWANID')


(3290, 29) (16142, 13)


## A. Baseline: 80/20 stratified split

Unsupervised clustering has no target to leak in the usual sense, but the held-out 20% is still useful —
clusters are learned on train, and test women are later assigned to those same clusters to check that the
structure generalises (Notebook 6+). The split is stratified on menopausal stage (`STATUS0`) purely so both
halves have a similar pre-/early-peri mix; `STATUS0` itself is never a feature.


In [2]:
status = df['STATUS0'].fillna('Unknown').replace({'Unknown due to HT use': 'Unknown'})
train_ids, test_ids = train_test_split(feat.index, test_size=0.20, random_state=RNG, stratify=status)
print(f"train = {len(train_ids):,}  test = {len(test_ids):,}")
print(pd.concat([status.loc[train_ids].value_counts(normalize=True).rename('train'),
                 status.loc[test_ids].value_counts(normalize=True).rename('test')], axis=1).round(3))


train = 2,632  test = 658
                train   test
STATUS0                     
Pre-menopausal  0.534  0.533
Early Peri      0.452  0.451
Unknown         0.014  0.015


In [3]:
json.dump({'train_ids': [int(i) for i in train_ids], 'test_ids': [int(i) for i in test_ids],
           'random_state': RNG, 'test_size': 0.20, 'stratified_on': 'STATUS0'},
          open(ART / 'split_ids.json', 'w'))
df[['STATUS0', 'RACE']].reset_index().assign(
    split=lambda d: np.where(d['SWANID'].isin(train_ids), 'train', 'test')
).to_csv(ART / 'baseline_posthoc_labels.csv', index=False)
print("Saved split_ids.json and baseline_posthoc_labels.csv")


Saved split_ids.json and baseline_posthoc_labels.csv


## B. Screener: building an independent validation cohort

Applied in order, reported at each step:
1. remove the 3,290 women already in the Baseline (cleaned) file — avoid double-counting;
2. keep only women reporting **no current hormone-therapy use** (`HORMONE == 'No'`) — comparable to Baseline,
   where every recorded HT/BCP flag was "No", and HT directly suppresses vasomotor symptoms;
3. drop women with >30% of the 11 symptom items missing (mirrors the Baseline row-drop rule from Notebook 2).


In [4]:
SCR_SYMP = ['HOTFLASH','NISWEAT','STIFF','HEADACH','FORGET','TENSE','DEPRESS','DRYNESS','IRRITAB','HEARTBO','URINE']
flow = [('Screener total', len(scr))]
v = scr.loc[~scr.index.isin(feat.index)];                      flow.append(('minus Baseline participants', len(v)))
v = v[v['HORMONE'] == 'No'];                                    flow.append(("current hormone use = 'No'", len(v)))
v = v[v[SCR_SYMP].isna().mean(axis=1) <= 0.30];                 flow.append(('<=30% symptoms missing', len(v)))
pd.DataFrame(flow, columns=['step', 'n women'])


,step,n women
0,Screener total,16142
1,minus Baseline participants,12852
2,current hormone use = 'No',8714
3,<=30% symptoms missing,8713


In [5]:
val_ids = v.index
scr_bmi = v['BMI'].where(v['BMI_FLAG'] == 'Body Mass Index (BMI) acceptable range')
post = pd.DataFrame({'STATUS': v['STATUS'].fillna('Unknown'), 'RACE': v['RACE'], 'AGE': v['AGE'], 'BMI': scr_bmi})
print("Validation cohort:", len(val_ids)); print(post['STATUS'].value_counts().to_string())

pd.Series(val_ids, name='SWANID').to_csv(ART / 'screener_validation_ids.csv', index=False)
post.to_csv(ART / 'screener_validation_posthoc.csv')
json.dump({'cohort_flow': flow, 'n_validation': int(len(val_ids))}, open(ART / 'screener_split_meta.json', 'w'), indent=2)
print("Saved screener_validation_ids.csv, screener_validation_posthoc.csv")


Validation cohort: 8713
STATUS
Premenopausal           2722
Early perimenopausal    1979
Unknown                 1672
Natural post            1513
Late perimenopausal      529
Undetermined             298
Saved screener_validation_ids.csv, screener_validation_posthoc.csv


### Summary

| Item | Value |
|---|---|
| Baseline train / test | stratified 80/20 on `STATUS0`, seed 42 |
| Screener validation cohort | independent of Baseline, current-HT users excluded, >30% missing excluded |
| Files saved | `split_ids.json`, `baseline_posthoc_labels.csv`, `screener_validation_ids.csv`, `screener_validation_posthoc.csv`, `screener_split_meta.json` |
| Next notebook | Notebook 5 — Preprocessing (impute / winsorise / scale, fit on train only) |
